In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from transformers.pipelines.pt_utils import KeyDataset
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from tqdm.auto import tqdm
import nltk
from nltk.corpus import stopwords
import warnings

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
def clean_text(text):
    # Lowercase (for uncased models)
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove URLs
    text = re.sub(r'(https?://\S+|www\.\S+)', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review'] = df['review'].apply(clean_text)

In [4]:
t5_df = df[['review', 'sentiment']].rename(columns={'sentiment': 'label', 'review': 'review'})
#t5_df = t5_df.groupby('label', group_keys=False).apply(lambda x: x.sample(100, random_state=42))

encoder = LabelEncoder()
t5_df['label'] = encoder.fit_transform(t5_df['label'])

reviews = t5_df['review'].tolist()
true_labels = t5_df['label'].tolist()

/tmp/ipykernel_55/1600725975.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t5_df = t5_df.groupby('label', group_keys=False).apply(lambda x: x.sample(100, random_state=42))


In [12]:
model_name = "google/flan-t5-base"  # "small" for smaller model
tokenizer = AutoTokenizer.from_pretrained(model_name, truncation=True, padding=True, max_length=512, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [8]:
prompts = {
    "simple": "Answer whether the following review is 'Positive' or 'Negative': '{}'",
    "detailed": "Read the following review carefully. Answer whether it is 'Positive' or 'Negative'. Here is the review: '{}'",
    "expert": "You are an expert in sentiment analysis. Analyze the sentiment of the following review and answer whether it is 'Positive' or 'Negative': '{}'"
}

In [ ]:
import torch
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

results = {"simple": [], "detailed": [], "expert": []}

for review in tqdm(reviews, desc="Processing reviews"):

   
    batch_prompts = [
        prompts["simple"].format(review),
        prompts["detailed"].format(review),
        prompts["expert"].format(review),
    ]

    encoded = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
        truncation_side="left"   
    )


    if "decoder_input_ids" in encoded:
        del encoded["decoder_input_ids"]


    encoded = {k: v.to(device) for k, v in encoded.items()}

   
    outputs = model.generate(
        input_ids=encoded["input_ids"],
        attention_mask=encoded["attention_mask"],
        max_new_tokens=3
    )


    decoded = [
        tokenizer.decode(o.cpu(), skip_special_tokens=True).lower()
        for o in outputs
    ]

    for key, resp in zip(results.keys(), decoded):
        if resp == "positive":
            predicted = encoder.transform(["positive"])[0]
        elif resp == "negative":
            predicted = encoder.transform(["negative"])[0]
        else:
            predicted = -1
        results[key].append(predicted)

In [8]:
import pandas as pd

# Tạo DataFrame kết quả
output_df = pd.DataFrame({
    "review": reviews,
    "true_label": true_labels,
    "pred_simple": results["simple"],
    "pred_detailed": results["detailed"],
    "pred_expert": results["expert"]
})

# ====== LƯU CSV ======
output_df.to_csv("t5_sentiment_results.csv", index=False, encoding="utf-8")

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

for key in results.keys():
    y_true = np.array(true_labels)
    y_pred = np.array(results[key])

    # Lọc bỏ -1
    mask = y_pred != -1
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    print(f"Classification report for {key} style:")
    print(classification_report(y_true, y_pred, target_names=encoder.classes_))
    print("-" * 50)

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_predictions(true_labels, pred_labels, label_names=["Negative", "Positive"]):


    pred_labels = [0 if p == -1 else p for p in pred_labels]

    acc = accuracy_score(true_labels, pred_labels)
    prec = precision_score(true_labels, pred_labels)
    rec = recall_score(true_labels, pred_labels)
    f1 = f1_score(true_labels, pred_labels)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print()


In [3]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/nguynnamhong1/t5-with-muti-prompt/t5_sentiment_results.csv")
df.head()

,review,true_label,pred_simple,pred_detailed,pred_expert
0,one of the other reviewers has mentioned that ...,1,1,1,1
1,a wonderful little production. the filming tec...,1,1,1,1
2,i thought this was a wonderful way to spend ti...,1,1,1,1
3,basically there's a family where a little boy ...,0,0,0,0
4,"petter mattei's ""love in the time of money"" is...",1,1,1,1


In [8]:
true = df["true_label"].values

print("===== SIMPLE =====")
evaluate_predictions(true, df["pred_simple"].values)

print("\n===== DETAILED =====")
evaluate_predictions(true, df["pred_detailed"].values)

print("\n===== EXPERT =====")
evaluate_predictions(true, df["pred_expert"].values)

===== SIMPLE =====
Accuracy : 0.9398
Precision: 0.9260
Recall   : 0.9560
F1-score : 0.9408


===== DETAILED =====
Accuracy : 0.9402
Precision: 0.9410
Recall   : 0.9392
F1-score : 0.9401


===== EXPERT =====
Accuracy : 0.9418
Precision: 0.9441
Recall   : 0.9395
F1-score : 0.9418
